In [1]:
import pandas as pd
df = pd.read_csv("phishing_data.csv")
print(df.columns.tolist())
print(df.head(1))

['Domain', 'Have_IP', 'Have_At', 'URL_Length', 'URL_Depth', 'Redirection', 'https_Domain', 'TinyURL', 'Prefix/Suffix', 'DNS_Record', 'Web_Traffic', 'Domain_Age', 'Domain_End', 'iFrame', 'Mouse_Over', 'Right_Click', 'Web_Forwards', 'Label']
             Domain  Have_IP  Have_At  URL_Length  URL_Depth  Redirection  \
0  graphicriver.net        0        0           1          1            0   

   https_Domain  TinyURL  Prefix/Suffix  DNS_Record  Web_Traffic  Domain_Age  \
0             0        0              0           0            1           1   

   Domain_End  iFrame  Mouse_Over  Right_Click  Web_Forwards  Label  
0           1       0           0            1             0      0  


In [27]:
import pandas as pd
import requests
import time

API_URL = "http://127.0.0.1:8000/scan"

def run_automated_stress_test(n=50):
    print(f"🧪 Q-SHIELD AUTOMATED VALIDATION")
    print(f"Target: {n} Random Samples (Unseen Data)")
    print("-" * 40)
    
    # Load and sample data
    df = pd.read_csv("phishing_data.csv")
    test_pool = df.iloc[3000:]  # Only use data the model hasn't 'seen' in the DB
    sample_df = test_pool.sample(n)
    
    feature_cols = ['Prefix/Suffix', 'URL_Length', 'Have_IP', 'Web_Traffic']
    
    stats = {"Safe_Correct": 0, "Safe_Total": 0, "Phish_Correct": 0, "Phish_Total": 0}
    start_time = time.time()

    for _, row in sample_df.iterrows():
        is_phish_actual = (str(row['Label']) == "1")
        
        # Track totals for breakdown
        if is_phish_actual: stats["Phish_Total"] += 1
        else: stats["Safe_Total"] += 1
            
        payload = {
            "url": row['Domain'], 
            "raw_features": row[feature_cols].astype(float).tolist()
        }
        
        try:
            res = requests.post(API_URL, json=payload).json()
            # Remember your Phase Correction logic: result['is_safe']
            if res['is_safe'] == (not is_phish_actual):
                if is_phish_actual: stats["Phish_Correct"] += 1
                else: stats["Safe_Correct"] += 1
        except Exception as e:
            print(f"⚠️ Connection Error: {e}")

    # Calculations
    total_correct = stats["Phish_Correct"] + stats["Safe_Correct"]
    accuracy = (total_correct / n) * 100
    duration = round(time.time() - start_time, 2)

    print(f"\n✅ VALIDATION COMPLETE ({duration}s)")
    print(f"📊 OVERALL ACCURACY: {accuracy:.2f}%")
    print(f"--- Breakdown ---")
    print(f"🟢 Safe Detection:    {stats['Safe_Correct']}/{stats['Safe_Total']}")
    print(f"🔴 Phish Detection:   {stats['Phish_Correct']}/{stats['Phish_Total']}")
    print("-" * 40)

if __name__ == "__main__":
    run_automated_stress_test(100)

🧪 Q-SHIELD AUTOMATED VALIDATION
Target: 100 Random Samples (Unseen Data)
----------------------------------------

✅ VALIDATION COMPLETE (1.39s)
📊 OVERALL ACCURACY: 82.00%
--- Breakdown ---
🟢 Safe Detection:    31/31
🔴 Phish Detection:   51/69
----------------------------------------
